Why Memory Is Needed Here

Without a checkpointer, every graph.invoke(...) starts with a blank messages list—if you ask "Multiply 5 by 2" and then ask "Now multiply that answer by 10", the graph forgets the 10.

By passing checkpointer=MemorySaver() when compiling the graph and passing config={"configurable": {"thread_id": "1"}} when invoking it, LangGraph saves the state after every step under that thread_id so follow-up questions remember the entire conversation.

In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

tavily_tool = TavilySearch(max_results=2)

@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

tools = [tavily_tool, multiply]

llm = init_chat_model("google_genai:gemini-3.5-flash-lite")
llm_with_tools = llm.bind_tools(tools)

In [2]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
)
builder.add_edge("tools", "tool_calling_llm")

# Initialize in-memory checkpointer and pass it into compile()
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

In [3]:
config = {"configurable": {"thread_id": "1"}}

response = graph.invoke(
    {"messages": [("user", "Hello, my name is Harshitha and please multiply 5 by 2")]},
    config=config,
)

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hello, my name is Harshitha and please multiply 5 by 2
================================== Ai Message ==================================

[]
Tool Calls:
  multiply (call_12456530436127557711)
 Call ID: call_12456530436127557711
  Args:
    b: 2
    a: 5
================================= Tool Message =================================
Name: multiply

10
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Hello Harshitha! 5 multiplied by 2 is 10.', 'extras': {'signature': 'EpYBCpMBAWkUfRPz9igXD5va1ckZOMSq8zqUx2B3YfffIdltweUfYq8TmnHlvJxbIM4TiGe/bcToMBAHIJnAQOmRHQ6DQGumCzmJPfuBSJY/rjD4CIxgQ+dOM8MleeKzbqs+SphWRv7rnl4GDsCbYjFxhXT47yCx3PeFIp9CLO2YNImW8t7DjryvEtrOOjmvkLiHi+8HL8L2'}}]


In [4]:
response = graph.invoke(
    {"messages": [("user", "What is my name and multiply my previous answer by 10?")]},
    config=config,
)

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hello, my name is Harshitha and please multiply 5 by 2
================================== Ai Message ==================================

[]
Tool Calls:
  multiply (call_12456530436127557711)
 Call ID: call_12456530436127557711
  Args:
    b: 2
    a: 5
================================= Tool Message =================================
Name: multiply

10
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Hello Harshitha! 5 multiplied by 2 is 10.', 'extras': {'signature': 'EpYBCpMBAWkUfRPz9igXD5va1ckZOMSq8zqUx2B3YfffIdltweUfYq8TmnHlvJxbIM4TiGe/bcToMBAHIJnAQOmRHQ6DQGumCzmJPfuBSJY/rjD4CIxgQ+dOM8MleeKzbqs+SphWRv7rnl4GDsCbYjFxhXT47yCx3PeFIp9CLO2YNImW8t7DjryvEtrOOjmvkLiHi+8HL8L2'}}]
================================ Human Message =================================

What is my name and multiply my previous answer by 10?
================================== Ai Mes

What Happens When You Run Chunk 4

Because both calls share config = {"configurable": {"thread_id": "1"}}, 
LangGraph loads the saved State from Chunk 3, sees your name (Harshitha) and the previous result (10), calls multiply(10, 10) to get 100, and answers both parts using past context. (If you change "thread_id" to "2", it starts a brand-new conversation with zero memory of thread "1".)